# 🎨 Paint-Code-RL: Zero-Cost GRPO Generative Art Training on Kaggle GPU

This notebook implements **Group Relative Policy Optimization (GRPO)** for training LLMs to write generative artwork in **p5.js** and **p5.brush**.

### 🚀 Key Capabilities:
- **Free Kaggle GPU Compute**: Runs natively on NVIDIA Tesla T4 (or P100) with CUDA acceleration.
- **Sandboxed WebGL Renderer**: Headless Chromium + SwiftShader/ANGLE WebGL daemon running on port 3000.
- **5-Tier Multi-Signal Visual RL Rewards**:
  1. Syntax & Structure (`setup()`, `createCanvas()`, `WEBGL`, `brush.scaleBrushes()`)
  2. Pixel-space Visual Richness (Color entropy & edge variance)
  3. Natural Media Brush Utilization (Watercolor washes, textures, hatching)
  4. Anti-Cheat Verifier (Penalizes canvas text injections)
  5. Aesthetic Scoring
- **Interactive Cyclic Training**: Unattended cycles, dynamic temperature annealing, and live dashboard visualization.

In [ ]:
# Cell 1: Check GPU Environment
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


In [ ]:
# Cell 2: Install System Dependencies for Headless Chromium WebGL
!apt-get update -qq
!apt-get install -y -qq chromium-browser nodejs npm xvfb > /dev/null 2>&1
!node -v && npm -v

In [ ]:
# Cell 3: Clone Repository and Checkout Branch
import os
if not os.path.exists("paint-code-rl"):
    !git clone https://github.com/harshitthek/paint-code-rl.git
%cd paint-code-rl
!git checkout feat/visual-rl-and-cyclic-training || git checkout main
!git pull origin feat/visual-rl-and-cyclic-training || true

In [ ]:
# Cell 4: Install Python RL Dependencies & Renderer Modules
!pip install -q trl==0.15.1 transformers==4.49.0 peft datasets accelerate pydantic safetensors Pillow pyyaml psutil requests
%cd renderer
!npm install --silent
%cd ..

In [ ]:
# Cell 5: Start Headless Node.js WebGL Renderer Daemon
import subprocess, time, requests

# Start daemon in background
proc = subprocess.Popen(["node", "renderer/server.js"])
time.sleep(3)

# Verify health
try:
    health = requests.get("http://127.0.0.1:3000/health", timeout=5).json()
    print("[OK] Renderer daemon is live:", health)
except Exception as e:
    print("[ERROR] Renderer failed to start:", e)

In [ ]:
# Cell 6: Run Fast Verification Tests (Unit & Security Suites)
!pytest tests/test_cyclic_and_scorecard.py tests/test_code_extractor_and_prompting.py tests/test_rewards.py -q

In [ ]:
# Cell 7: Launch Unattended GRPO Cyclic Training on GPU
import os
os.environ["ENV"] = "kaggle"
os.environ["PYTHONUNBUFFERED"] = "1"

# Run 100 training steps (4 cycles of 25 steps) with auto hardware saturation & live dashboard
!python scripts/train_grpo.py --mode train --steps-per-cycle 25 --max-steps 100 --unattended --max --dashboard

In [ ]:
# Cell 8: Display Live Dashboard & Generated Art Gallery
import glob
from IPython.display import display, HTML, Image

# Check generated renders
render_files = sorted(glob.glob("artifacts/renders/*.png"), reverse=True)
print(f"Total artworks rendered: {len(render_files)}")

for img_path in render_files[:6]:
    print(f"Artwork: {img_path}")
    display(Image(filename=img_path, width=400))

# Render live dashboard inline
if os.path.exists("artifacts/dashboard.html"):
    with open("artifacts/dashboard.html", "r", encoding="utf-8") as f:
        html_code = f.read()
    display(HTML(f'<iframe srcdoc="{html_code.replace(chr(34), "&quot;")}" width="100%" height="600px" frameborder="0"></iframe>'))

In [ ]:
# Cell 9: Generate New Artwork from Custom User Prompts
!python scripts/generate_and_render.py --output-dir artifacts/custom_renders

custom_artworks = sorted(glob.glob("artifacts/custom_renders/*.png"))
for art in custom_artworks:
    print(f"Custom Generated Art: {art}")
    display(Image(filename=art, width=400))